In [1]:
# 01_kafka_profiling — Load Bronze Events & Run Streaming Profiling
# -----------------------------------------------------------------
from scripts.storage.config.paths import get_path_builder, SourceSystem
from scripts.storage.config.spark import get_spark_session
from pyspark.sql import DataFrame, functions as F
from pyspark.sql.types import TimestampType, StringType

spark = get_spark_session(app_name="kafka_silver_eda")
path_builder = get_path_builder()

try:
    import dbutils  # type: ignore
    ingestion_date = dbutils.widgets.get("ingestion_date")
except Exception:
    ingestion_date = "2026-11-15"

print(f"Spark Session: {spark.sparkContext.appName}")
print(f"Ingestion Date: {ingestion_date}")

kafka_bronze_path = path_builder.bronze_kafka_streaming(ingestion_date)
print(f"Bronze Path: {kafka_bronze_path}")

df_kafka = spark.read.parquet(kafka_bronze_path)
total_events = df_kafka.count()
print(f"Loaded events: {total_events:,} rows x {len(df_kafka.columns)} cols")

print("\n" + "=" * 80)
print("SCHEMA — KAFKA EVENTS")
print("=" * 80)
df_kafka.printSchema()

print("\n" + "=" * 80)
print("TIMESTAMP CORRUPTION ANALYSIS")
print("=" * 80)

timestamp_col = "event_timestamp"
if timestamp_col in df_kafka.columns:
    orig_nulls = df_kafka.filter(F.col(timestamp_col).isNull()).count()
    
    df_ts_check = df_kafka.withColumn(
        "ts_parsed", F.col(timestamp_col).cast(TimestampType())
    )
    corrupted = df_ts_check.filter(
        F.col(timestamp_col).isNotNull() & F.col("ts_parsed").isNull()
    )
    corrupted_count = corrupted.count()
    
    print(f"Original null timestamps: {orig_nulls:,}")
    print(f"Corrupted timestamps (unparseable): {corrupted_count:,}")
    
    if corrupted_count > 0:
        print("\nSample of corrupted timestamp values:")
        corrupted.select(timestamp_col).distinct().show(10, truncate=False)
    
    valid_range = df_ts_check.filter(F.col("ts_parsed").isNotNull()).agg(
        F.min("ts_parsed").alias("min_ts"),
        F.max("ts_parsed").alias("max_ts")
    ).collect()[0]
    print(f"\nValid timestamp range: {valid_range['min_ts']} to {valid_range['max_ts']}")
else:
    print(f"Column '{timestamp_col}' not found in schema")

print("\n" + "=" * 80)
print("WATCH-TIME BUSINESS RULE VALIDATION")
print("=" * 80)

watch_col = "watch_time_sec"
interaction_col = "interaction_type"

if watch_col in df_kafka.columns and interaction_col in df_kafka.columns:
    play_complete_null_wt = df_kafka.filter(
        F.col(interaction_col).isin("play", "complete") & F.col(watch_col).isNull()
    ).count()
    print(f"play/complete with NULL watch_time: {play_complete_null_wt:,}")
    
    like_pause_with_wt = df_kafka.filter(
        F.col(interaction_col).isin("like", "pause") & F.col(watch_col).isNotNull()
    ).count()
    print(f"like/pause with non-NULL watch_time: {like_pause_with_wt:,}")
    
    wt_le_zero = df_kafka.filter(
        F.col(watch_col).isNotNull() & (F.col(watch_col) <= 0)
    ).count()
    print(f"watch_time <= 0: {wt_le_zero:,}")
    
    wt_gt_120 = df_kafka.filter(
        F.col(watch_col).isNotNull() & (F.col(watch_col) > 120)
    ).count()
    print(f"watch_time > 120s: {wt_gt_120:,}")
    
    wt_stats_row = df_kafka.filter(F.col(watch_col).isNotNull()).agg(
        F.min(F.col(watch_col).cast("double")).alias("min"),
        F.max(F.col(watch_col).cast("double")).alias("max"),
        F.mean(F.col(watch_col).cast("double")).alias("mean"),
        F.stddev(F.col(watch_col).cast("double")).alias("stddev"),
        F.expr(f"percentile_approx(cast({watch_col} as double), 0.5)").alias("median"),
        F.expr(f"percentile_approx(cast({watch_col} as double), 0.95)").alias("p95"),
        F.expr(f"percentile_approx(cast({watch_col} as double), 0.99)").alias("p99")
    ).collect()[0]
    
    print(f"\nWatch-time statistics (non-null):")
    print(f"  Min: {wt_stats_row['min']}, Max: {wt_stats_row['max']}, Mean: {wt_stats_row['mean']:.2f}")
    print(f"  StdDev: {wt_stats_row['stddev']:.2f}, Median: {wt_stats_row['median']}")
    print(f"  P95: {wt_stats_row['p95']}, P99: {wt_stats_row['p99']}")
else:
    print("Required columns (watch_time_sec, interaction_type) not found")

print("\n" + "=" * 80)
print("CARDINALITY & DISTRIBUTION ANALYSIS")
print("=" * 80)

if interaction_col in df_kafka.columns:
    print(f"\n{interaction_col} distribution:")
    df_kafka.groupBy(interaction_col).count().orderBy(F.desc("count")).show(truncate=False)

device_col = "device_type"
if device_col in df_kafka.columns:
    print(f"\n{device_col} distribution:")
    df_kafka.groupBy(device_col).count().orderBy(F.desc("count")).show(truncate=False)

event_id_col = "event_id"
if event_id_col in df_kafka.columns:
    total = total_events
    distinct_eid = df_kafka.select(event_id_col).distinct().count()
    dup_groups = df_kafka.groupBy(event_id_col).count().filter(F.col("count") > 1).count()
    print(f"\n{event_id_col} uniqueness:")
    print(f"  Total events: {total:,}")
    print(f"  Distinct event_ids: {distinct_eid:,}")
    print(f"  Duplicate event_id groups: {dup_groups:,}")
    if dup_groups > 0:
        print("  Sample duplicate event_ids:")
        df_kafka.groupBy(event_id_col).count().filter(F.col("count") > 1).orderBy(F.desc("count")).show(10, truncate=False)

for id_col in ["user_id", "video_id"]:
    if id_col in df_kafka.columns:
        nulls = df_kafka.filter(F.col(id_col).isNull()).count()
        empty = df_kafka.filter(F.trim(F.col(id_col)) == "").count()
        distinct = df_kafka.select(id_col).distinct().count()
        print(f"\n{id_col} quality:")
        print(f"  Nulls: {nulls:,}, Empty strings: {empty:,}, Distinct: {distinct:,}")

print("\n" + "=" * 80)
print("KAFKA EDA COMPLETE — NO TRANSFORMATIONS PERFORMED")
print("=" * 80)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/15 12:27:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session: kafka_silver_eda
Ingestion Date: 2026-11-15
Bronze Path: s3a://end-to-end-streaming-data-platform-bronze/kafka/ingestion_date=2026-11-15


26/11/15 12:27:10 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

Loaded events: 9,852 rows x 7 cols

SCHEMA — KAFKA EVENTS
root
 |-- event_id: string (nullable = true)
 |-- event_timestamp: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- video_id: string (nullable = true)
 |-- interaction_type: string (nullable = true)
 |-- watch_time_sec: integer (nullable = true)
 |-- device_type: string (nullable = true)


TIMESTAMP CORRUPTION ANALYSIS
Original null timestamps: 0
Corrupted timestamps (unparseable): 0

Valid timestamp range: 2025-12-31 23:59:59 to 2027-01-01 00:00:00

WATCH-TIME BUSINESS RULE VALIDATION
play/complete with NULL watch_time: 985
like/pause with non-NULL watch_time: 452
watch_time <= 0: 867
watch_time > 120s: 1,182



Watch-time statistics (non-null):
  Min: 0.0, Max: 10000.0, Mean: 98.96
  StdDev: 763.87, Median: 1.0
  P95: 252.0, P99: 355.0

CARDINALITY & DISTRIBUTION ANALYSIS

interaction_type distribution:
+----------------+-----+
|interaction_type|count|
+----------------+-----+
|play            |6896 |
|complete        |1454 |
|pause           |1324 |
|like            |118  |
|INVALID_TYPE    |60   |
+----------------+-----+


device_type distribution:
+--------------+-----+
|device_type   |count|
+--------------+-----+
|mobile        |5062 |
|tv            |3296 |
|web           |1455 |
|INVALID_DEVICE|39   |
+--------------+-----+


event_id uniqueness:
  Total events: 9,852
  Distinct event_ids: 9,852
  Duplicate event_id groups: 0

user_id quality:
  Nulls: 0, Empty strings: 0, Distinct: 3,544

video_id quality:
  Nulls: 0, Empty strings: 0, Distinct: 3,316

KAFKA EDA COMPLETE — NO TRANSFORMATIONS PERFORMED


In [2]:
# 01_kafka_profiling — Load Bronze Events & Run Streaming Profiling
# -----------------------------------------------------------------
from scripts.storage.config.paths import get_path_builder, SourceSystem
from scripts.storage.config.spark import get_spark_session
from pyspark.sql import DataFrame, functions as F
from pyspark.sql.types import TimestampType, StringType

spark = get_spark_session(app_name="kafka_silver_eda")
path_builder = get_path_builder()

try:
    import dbutils  # type: ignore
    ingestion_date = dbutils.widgets.get("ingestion_date")
except Exception:
    ingestion_date = "2026-11-15"

print(f"Spark Session: {spark.sparkContext.appName}")
print(f"Ingestion Date: {ingestion_date}")

kafka_bronze_path = path_builder.bronze_kafka_streaming(ingestion_date)
print(f"Bronze Path: {kafka_bronze_path}")

df_kafka = spark.read.parquet(kafka_bronze_path)
total_events = df_kafka.count()
print(f"Loaded events: {total_events:,} rows x {len(df_kafka.columns)} cols")
# =============================================================================
# SUPPLEMENTARY: DIRTY DATA INJECTION PROFILING — KAFKA EVENTS
# Targets anomalies from KafkaDirtyDataInjector._inject_event_anomalies()
# =============================================================================

print("=" * 80)
print("DIRTY DATA INJECTION PROFILING — KAFKA EVENTS")
print("=" * 80)

# ---------------------------------------------------------------
# Anomaly Type 1: Null mandatory field — watch_time_sec
# ---------------------------------------------------------------
null_watch_time = df_kafka.filter(F.col("watch_time_sec").isNull()).count()
print(f"\n[Anomaly 1] Null watch_time_sec: {null_watch_time:,}")

# Breakdown by interaction_type for null watch_time
if null_watch_time > 0:
    df_kafka.filter(F.col("watch_time_sec").isNull()) \
        .groupBy("interaction_type").count() \
        .orderBy(F.desc("count")).show(truncate=False)

# ---------------------------------------------------------------
# Anomaly Type 2: Invalid interaction_type enum ("INVALID_TYPE")
# ---------------------------------------------------------------
invalid_interaction = df_kafka.filter(
    F.col("interaction_type") == "INVALID_TYPE"
).count()
print(f"\n[Anomaly 2] Invalid interaction_type='INVALID_TYPE': {invalid_interaction:,}")

# All non-standard interaction values
print("\n[Anomaly 2] Full interaction_type value distribution (including dirty):")
df_kafka.groupBy("interaction_type").count().orderBy(F.desc("count")).show(truncate=False)

# ---------------------------------------------------------------
# Anomaly Type 3 & 4: Temporal skew — event_timestamp outside simulation window
# Simulation window: 2026-01-01 to 2026-12-31
# ---------------------------------------------------------------
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType

df_ts = df_kafka.withColumn("ts_parsed", F.col("event_timestamp").cast(TimestampType()))

# Before simulation start (2025-12-31 23:59:59)
skew_before = df_ts.filter(
    F.col("ts_parsed").isNotNull() & 
    (F.col("ts_parsed") < F.lit("2026-01-01 00:00:00"))
).count()
print(f"\n[Anomaly 3] Timestamp before simulation start (2026-01-01): {skew_before:,}")

# After simulation end (2027-01-01 00:00:00)
skew_after = df_ts.filter(
    F.col("ts_parsed").isNotNull() & 
    (F.col("ts_parsed") >= F.lit("2027-01-01 00:00:00"))
).count()
print(f"[Anomaly 4] Timestamp after simulation end (2026-12-31): {skew_after:,}")

# Show sample corrupted timestamps
if skew_before > 0 or skew_after > 0:
    print("\nSample out-of-bounds timestamps:")
    df_ts.filter(
        (F.col("ts_parsed") < F.lit("2026-01-01 00:00:00")) |
        (F.col("ts_parsed") >= F.lit("2027-01-01 00:00:00"))
    ).select("event_timestamp", "ts_parsed").distinct().show(20, truncate=False)

# ---------------------------------------------------------------
# Anomaly Type 5: Invalid device_type enum ("INVALID_DEVICE")
# ---------------------------------------------------------------
invalid_device = df_kafka.filter(
    F.col("device_type") == "INVALID_DEVICE"
).count()
print(f"\n[Anomaly 5] Invalid device_type='INVALID_DEVICE': {invalid_device:,}")

print("\n[Anomaly 5] Full device_type value distribution (including dirty):")
df_kafka.groupBy("device_type").count().orderBy(F.desc("count")).show(truncate=False)

# ---------------------------------------------------------------
# Anomaly Type 6: Metric invariant break — watch_time_sec > video duration (play events)
# Injector sets watch_time_sec = 10000 for 'play' interactions
# ---------------------------------------------------------------
# Note: This requires video duration lookup. Profiling extreme watch_time values:
extreme_watch_time = df_kafka.filter(
    F.col("watch_time_sec").isNotNull() & (F.col("watch_time_sec") > 7200)  # > 2 hours
).count()
print(f"\n[Anomaly 6] Extreme watch_time_sec (> 7200s / 2hrs): {extreme_watch_time:,}")

if extreme_watch_time > 0:
    print("\nSample extreme watch_time values:")
    df_kafka.filter(F.col("watch_time_sec") > 7200) \
        .select("event_id", "interaction_type", "watch_time_sec", "video_id") \
        .show(20, truncate=False)

# Watch time distribution for play events specifically
play_watch_stats = df_kafka.filter(F.col("interaction_type") == "play") \
    .agg(
        F.min("watch_time_sec").alias("min"),
        F.max("watch_time_sec").alias("max"),
        F.mean("watch_time_sec").alias("mean"),
        F.expr("percentile_approx(watch_time_sec, 0.99)").alias("p99")
    ).collect()[0]
print(f"\nPlay events watch_time stats: min={play_watch_stats['min']}, max={play_watch_stats['max']}, mean={play_watch_stats['mean']:.1f}, p99={play_watch_stats['p99']}")

# ---------------------------------------------------------------
# DISABLED Anomaly (Type 7): Referential orphan — user_id/video_id mutation
# Currently disabled in injector, but verify no empty strings exist
# ---------------------------------------------------------------
empty_user_id = df_kafka.filter(F.trim(F.col("user_id")) == "").count()
empty_video_id = df_kafka.filter(F.trim(F.col("video_id")) == "").count()
print(f"\n[Disabled Anomaly 7 Check] Empty user_id: {empty_user_id:,}, Empty video_id: {empty_video_id:,}")

# ---------------------------------------------------------------
# SUMMARY: Blended anomaly rate estimation
# ---------------------------------------------------------------
total_events = df_kafka.count()
# Approximate: each anomaly type has ~1/6 probability when injection triggers
# Injection rate ~3.5% -> each anomaly ~0.58% of total
print(f"\n{'=' * 80}")
print(f"ESTIMATED ANOMALY COUNTS (at ~3.5% injection rate, 6 types):")
print(f"  Expected per anomaly type: ~{total_events * 0.035 / 6:,.0f} rows")
print(f"  Total expected dirty rows: ~{total_events * 0.035:,.0f} ({3.5}%)")
print(f"{'=' * 80}")


Spark Session: kafka_silver_eda
Ingestion Date: 2026-11-15
Bronze Path: s3a://end-to-end-streaming-data-platform-bronze/kafka/ingestion_date=2026-11-15
Loaded events: 9,852 rows x 7 cols
DIRTY DATA INJECTION PROFILING — KAFKA EVENTS

[Anomaly 1] Null watch_time_sec: 1,988
+----------------+-----+
|interaction_type|count|
+----------------+-----+
|complete        |950  |
|pause           |872  |
|like            |118  |
|play            |35   |
|INVALID_TYPE    |13   |
+----------------+-----+


[Anomaly 2] Invalid interaction_type='INVALID_TYPE': 60

[Anomaly 2] Full interaction_type value distribution (including dirty):
+----------------+-----+
|interaction_type|count|
+----------------+-----+
|play            |6896 |
|complete        |1454 |
|pause           |1324 |
|like            |118  |
|INVALID_TYPE    |60   |
+----------------+-----+


[Anomaly 3] Timestamp before simulation start (2026-01-01): 62
[Anomaly 4] Timestamp after simulation end (2026-12-31): 61

Sample out-of-bounds

In [3]:
# 01_kafka_profiling — Load Bronze Events & Run Streaming Profiling
# -----------------------------------------------------------------
from scripts.storage.config.paths import get_path_builder, SourceSystem
from scripts.storage.config.spark import get_spark_session
from pyspark.sql import DataFrame, functions as F
from pyspark.sql.types import TimestampType, StringType

spark = get_spark_session(app_name="kafka_silver_eda")
path_builder = get_path_builder()

try:
    import dbutils  # type: ignore
    ingestion_date = dbutils.widgets.get("ingestion_date")
except Exception:
    ingestion_date = "2026-11-15"

print(f"Spark Session: {spark.sparkContext.appName}")
print(f"Ingestion Date: {ingestion_date}")

kafka_bronze_path = path_builder.bronze_kafka_streaming(ingestion_date)
print(f"Bronze Path: {kafka_bronze_path}")

df_kafka = spark.read.parquet(kafka_bronze_path)
total_events = df_kafka.count()
print(f"Loaded events: {total_events:,} rows x {len(df_kafka.columns)} cols")
# ---------------------------------------------------------------
# Anomaly Type 3 & 4: Temporal skew — event_timestamp outside simulation window
# Simulation window: 2026-01-01 to 2026-12-31 (inclusive)
# Injector constants: 2025-12-31 23:59:59 (pre) and 2027-01-01 00:00:00 (post)
# ---------------------------------------------------------------
from pyspark.sql.types import TimestampType

df_ts = df_kafka.withColumn("ts_parsed", F.col("event_timestamp").cast(TimestampType()))

# Pre-simulation (before 2026-01-01)
skew_pre = df_ts.filter(
    F.col("ts_parsed").isNotNull() & 
    (F.col("ts_parsed") < F.lit("2026-01-01 00:00:00"))
).count()
print(f"\n[Anomaly 3] Timestamp before simulation start (2026-01-01): {skew_pre:,}")

# Post-simulation (after 2026-12-31 23:59:59, i.e., >= 2027-01-01)
skew_post = df_ts.filter(
    F.col("ts_parsed").isNotNull() & 
    (F.col("ts_parsed") >= F.lit("2027-01-01 00:00:00"))
).count()
print(f"[Anomaly 4] Timestamp after simulation end (2026-12-31): {skew_post:,}")

if skew_pre > 0 or skew_post > 0:
    print("\nSample out-of-bounds timestamps:")
    df_ts.filter(
        (F.col("ts_parsed") < F.lit("2026-01-01 00:00:00")) |
        (F.col("ts_parsed") >= F.lit("2027-01-01 00:00:00"))
    ).select("event_timestamp", "ts_parsed").distinct().show(20, truncate=False)


Spark Session: kafka_silver_eda
Ingestion Date: 2026-11-15
Bronze Path: s3a://end-to-end-streaming-data-platform-bronze/kafka/ingestion_date=2026-11-15
Loaded events: 9,852 rows x 7 cols

[Anomaly 3] Timestamp before simulation start (2026-01-01): 62
[Anomaly 4] Timestamp after simulation end (2026-12-31): 61

Sample out-of-bounds timestamps:
+-------------------------+-------------------+
|event_timestamp          |ts_parsed          |
+-------------------------+-------------------+
|2025-12-31 23:59:59+00:00|2025-12-31 23:59:59|
|2027-01-01 00:00:00+00:00|2027-01-01 00:00:00|
+-------------------------+-------------------+

